# Hexagon Segmentation Pipeline

Reads `02_merged_data.parquet` and assigns H3 hexagons at a configurable resolution

- `H3_RESOLUTION` — hexagon size (0 = coarsest, 15 = finest; 7≈5 km, 8≈1 km, 9≈0.3 km)

In [1]:
import pandas as pd
import polars as pl
import h3

## Load data

In [2]:
df = pl.read_parquet("../data/02/02_merged_data.parquet").to_pandas()
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
# df.head(3)

Shape: 13,478,292 rows x 51 columns


In [3]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds',
       'trip_miles', 'pickup_census_tract', 'dropoff_census_tract',
       'pickup_community_area', 'dropoff_community_area', 'fare_usd',
       'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type',
       'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
       'average_speed_mph', 'hour', 'day_of_week', 'month', 'date',
       'is_weekend', 'bin_30min', 'bin_1h', 'bin_4h', 'bin_1d', 'bin_1w',
       'date_day', 'DayOfWeek_Friday', 'DayOfWeek_Saturday',
       'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday',
       'DayOfWeek_Wednesday', 'BankHoliday', 'CorporateHoliday', 'CityHoliday',
       'StateHoliday', 'temperature_c', 'temperature_max_c',
       'temperature_min_c', 'rain', 'strong_rain', 'snow', 'wind',
       'strong_wind'],
      dtype='str')

# Assign H3 Hexagons

In [4]:
def to_h3_cell(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    return h3.latlng_to_cell(float(lat), float(lon), 9)

df["start_h3_r9"] = [to_h3_cell(lat, lon) for lat, lon in zip(df["pickup_lat"],  df["pickup_lon"])]
df["end_h3_r9"]   = [to_h3_cell(lat, lon) for lat, lon in zip(df["dropoff_lat"], df["dropoff_lon"])]

print(f"  R9: {df['start_h3_r9'].nunique():>4} pickup cells, {df['end_h3_r9'].nunique():>4} dropoff cells")


  R9:  537 pickup cells,  663 dropoff cells


In [5]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds',
       'trip_miles', 'pickup_census_tract', 'dropoff_census_tract',
       'pickup_community_area', 'dropoff_community_area', 'fare_usd',
       'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type',
       'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
       'average_speed_mph', 'hour', 'day_of_week', 'month', 'date',
       'is_weekend', 'bin_30min', 'bin_1h', 'bin_4h', 'bin_1d', 'bin_1w',
       'date_day', 'DayOfWeek_Friday', 'DayOfWeek_Saturday',
       'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday',
       'DayOfWeek_Wednesday', 'BankHoliday', 'CorporateHoliday', 'CityHoliday',
       'StateHoliday', 'temperature_c', 'temperature_max_c',
       'temperature_min_c', 'rain', 'strong_rain', 'snow', 'wind',
       'strong_wind', 'start_h3_r9', 'end_h3_r9'],
      dtype='str')

# Save Parquet

In [6]:
df.to_parquet("../data/03/03_merged_data_sample_10pct_with_h3.parquet", index=False)